# MLP 모델

이 노트북은 `src/models/mlp.py`의 `MLP`를 직접 실행하고 forward/backward/파라미터 업데이트 흐름을 검증하는 실습 자료이다.
이전 노트북의 변수나 실행 결과를 사용하지 않으며, 이 노트북 단독으로 완전히 실행할 수 있다.

**실행 환경**: `conda run -n numpy_py311` (GPU 불필요)

**목표**
- `MLP` 생성자와 seed 파생 방식을 확인한다.
- task별 logit shape와 파라미터 구성(`params`/`grads`)을 검증한다.
- forward → loss → backward → optimizer.step 한 스텝 흐름을 수동으로 실행한다.
- 수동 SGD로 학습 곡선을 시각화하여 loss가 감소하는 것을 확인한다.

## 0. 환경 설정

In [ ]:
# sys.path setup -- excluded from jupyter book build
import os
import sys

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print(f"project_root={PROJECT_ROOT}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
from src.models.mlp import MLP
from src.nn.losses import (
    cross_entropy, cross_entropy_grad,
    binary_cross_entropy, binary_cross_entropy_grad,
    mse, mse_grad,
)
from src.nn.metrics import accuracy, binary_accuracy, r2_score

## 1. 개요

`MLP`는 `src/nn/` 레이어 모듈을 조립하여 구성한 3층 완전 연결 네트워크이다.
`forward`는 raw logit을 반환하고, activation과 gradient 계산은 `src/nn/losses.py`가 처리한다.

**구조**: `784 → 256 → 128 → output_dim`

```
입력: (N, 784)
  Linear(784, 256) → Sigmoid
  Linear(256, 128) → Sigmoid
  Linear(128, output_dim)
출력: (N, output_dim)  ← raw logit
```

task별 출력 설정은 다음과 같다.

| task | output_dim | 출력 activation | 손실 함수 |
|---|---|---|---|
| `multiclass` | 10 | `softmax` (losses.py 내부) | `cross_entropy` |
| `binary` | 1 | `sigmoid` (losses.py 내부) | `binary_cross_entropy` |
| `regression` | 1 | 없음 (identity) | `mse` |

## 2. 모델 생성과 파라미터 구성

`MLP`는 최상위 `seed` 하나를 받아 3개의 레이어 seed를 파생한다.
최상위 seed 하나로 전체 초기화의 재현성이 보장된다.

`MLP.params = [W1, b1, W2, b2, W3, b3]` (총 6개 배열)

In [ ]:
model = MLP(task="multiclass", seed=42)

print(f"task:       {model.task}")
print(f"output_dim: {model.output_dim}")
print(f"params 수:  {len(model.params)}  <- Linear 3개 × (W, b) = 6")
print(f"grads  수:  {len(model.grads)}")
print()
print("=== 파라미터 shape ===")
names = ["W1", "b1", "W2", "b2", "W3", "b3"]
for name, p in zip(names, model.params):
    print(f"  {name}: shape={p.shape}, dtype={p.dtype}")

In [ ]:
# 총 파라미터 수 계산
total = sum(p.size for p in model.params)
print(f"총 파라미터 수: {total:,}")
print(f"메모리 (float32): {total * 4 / 1024**2:.2f} MB")
print()
# 레이어별 분해
print("레이어별 파라미터 수:")
for i in range(0, len(model.params), 2):
    w, b = model.params[i], model.params[i+1]
    n = i // 2 + 1
    print(f"  Linear {n}: W={w.shape} ({w.size:,}) + b={b.shape} ({b.size}) = {w.size + b.size:,}")

## 3. Seed 재현성

동일한 seed는 동일한 초기 파라미터를 생성하고, 다른 seed는 다른 초기 파라미터를 생성한다.

In [ ]:
model_a = MLP(task="multiclass", seed=42)
model_b = MLP(task="multiclass", seed=42)
model_c = MLP(task="multiclass", seed=99)

same = np.allclose(model_a.params[0], model_b.params[0])
diff = not np.allclose(model_a.params[0], model_c.params[0])
print(f"seed=42 vs seed=42: W1 동일 → {same}  (기대: True)")
print(f"seed=42 vs seed=99: W1 다름 → {diff}  (기대: True)")

## 4. Forward — task별 logit shape

`forward(x)`는 `(N, output_dim)` raw logit을 반환한다.
activation은 적용하지 않으며, losses.py가 내부에서 처리한다.

In [ ]:
N = 32
x = np.random.randn(N, 784).astype(np.float32)

for task, expected_dim in [("multiclass", 10), ("binary", 1), ("regression", 1)]:
    m = MLP(task=task, seed=0)
    logits = m.forward(x)
    print(f"task={task:<12}: logits.shape={logits.shape}, dtype={logits.dtype}")
    assert logits.shape == (N, expected_dim)
    assert logits.dtype == np.float32

## 5. Backward — gradient 계산

`model.backward(grad_out)`은 `losses.*_grad`가 반환한 `d(loss)/d(logits)`를 출발점으로
각 레이어를 역순으로 순회하며 파라미터 gradient를 `grads` 리스트에 저장한다.

In [ ]:
model = MLP(task="multiclass", seed=42)
N = 16
x = np.random.randn(N, 784).astype(np.float32)
targets = np.eye(10, dtype=np.float32)[np.arange(N) % 10]

# forward
logits = model.forward(x)
loss = cross_entropy(logits, targets)
grad_out = cross_entropy_grad(logits, targets)

# grads는 backward 전에 0이어야 한다
all_zero_before = all(np.all(g == 0) for g in model.grads)
print(f"backward 전 모든 grads=0: {all_zero_before}")

# backward
model.backward(grad_out)

# backward 후 grads가 0이 아니어야 한다
print(f"\nloss: {loss:.4f}")
print(f"grad_out shape: {grad_out.shape}")
print()
print("=== backward 후 gradient norm ===")
for name, g in zip(names, model.grads):
    print(f"  grads[{name}]: norm={np.linalg.norm(g):.4f}, nonzero={np.count_nonzero(g)}")

In [ ]:
# params는 backward 후에도 변경되지 않아야 한다 (optimizer.step() 전)
w1_before = model.params[0].copy()
model.backward(grad_out)  # 한 번 더 실행해도 params 불변
print(f"backward 후 params[W1] 불변: {np.allclose(w1_before, model.params[0])}")
print("→ backward는 grads만 업데이트하고 params는 수정하지 않는다")

## 6. 학습 한 스텝 흐름

forward → loss → backward → optimizer.step 한 스텝을 수동으로 실행한다.

```
[1] images, targets = next(train_loader)
[2] logits = model.forward(images)        # (N, output_dim) raw logit
[3] loss   = cross_entropy(logits, targets)
[4] grad   = cross_entropy_grad(logits, targets)   # d(loss)/d(logits)
[5] model.backward(grad)                  # 각 레이어 grad_w, grad_b 계산
[6] for p, g in zip(model.params, model.grads): p -= lr * g
```

In [ ]:
model = MLP(task="multiclass", seed=42)
lr = 0.01
N = 32

x = np.random.randn(N, 784).astype(np.float32)
targets = np.eye(10, dtype=np.float32)[np.arange(N) % 10]

w1_before = model.params[0].copy()

# 1 step
logits = model.forward(x)
loss_before = cross_entropy(logits, targets)
grad_out = cross_entropy_grad(logits, targets)
model.backward(grad_out)
for p, g in zip(model.params, model.grads):
    p -= lr * g

# params 업데이트 확인
w1_after = model.params[0]
print(f"W1 변경량 (|Δw|): {np.abs(w1_after - w1_before).mean():.6f}")
print(f"W1 변경됨: {not np.allclose(w1_before, w1_after)}")

# 업데이트 후 loss 확인
logits_after = model.forward(x)
loss_after = cross_entropy(logits_after, targets)
print(f"\nloss 전: {loss_before:.4f}")
print(f"loss 후: {loss_after:.4f}  ({'감소' if loss_after < loss_before else '증가'})")

## 7. 수동 학습 곡선

3가지 task에 대해 소규모 synthetic 데이터로 수동 SGD를 실행하여 학습 곡선을 시각화한다.

In [ ]:
np.random.seed(0)
N_train = 64
n_steps = 100
lr = 0.05

configs = [
    ("multiclass", cross_entropy,        cross_entropy_grad,        accuracy),
    ("binary",     binary_cross_entropy,  binary_cross_entropy_grad,  binary_accuracy),
    ("regression", mse,                   mse_grad,                   r2_score),
]

fig, axes = plt.subplots(2, 3, figsize=(14, 7))

for col, (task, loss_fn, grad_fn, metric_fn) in enumerate(configs):
    model = MLP(task=task, seed=42)

    # synthetic data
    x_syn = np.random.randn(N_train, 784).astype(np.float32)
    if task == "multiclass":
        y_syn = np.eye(10, dtype=np.float32)[np.arange(N_train) % 10]
    elif task == "binary":
        y_syn = (np.arange(N_train) % 2).reshape(-1, 1).astype(np.float32)
    else:
        y_syn = np.random.rand(N_train, 1).astype(np.float32)

    losses, metrics = [], []
    for _ in range(n_steps):
        logits = model.forward(x_syn)
        losses.append(float(loss_fn(logits, y_syn)))
        metrics.append(float(metric_fn(logits, y_syn)))
        grad = grad_fn(logits, y_syn)
        model.backward(grad)
        for p, g in zip(model.params, model.grads):
            p -= lr * g

    axes[0, col].plot(losses, color='steelblue', linewidth=2)
    axes[0, col].set_title(f"{task}\nloss")
    axes[0, col].set_xlabel("step")
    axes[0, col].grid(alpha=0.3)

    axes[1, col].plot(metrics, color='tomato', linewidth=2)
    metric_name = {"multiclass": "accuracy", "binary": "binary_accuracy", "regression": "r2_score"}[task]
    axes[1, col].set_title(f"{task}\n{metric_name}")
    axes[1, col].set_xlabel("step")
    axes[1, col].grid(alpha=0.3)

    print(f"{task:<12}: loss {losses[0]:.3f} → {losses[-1]:.3f}")

fig.suptitle("MLP — Manual SGD Training Curves", fontsize=13)
fig.tight_layout()
plt.show()

## 8. He 초기화 확인

각 레이어의 가중치 표준편차가 He 초기화 기준에 가까운지 확인한다.

$$
\text{std} \approx \sqrt{\frac{2}{D_{in}}}
$$

In [ ]:
model = MLP(task="multiclass", seed=42)
in_dims = [784, 256, 128]

print(f"{'레이어':<15} {'실제 std':<12} {'He 기준':<12} {'비율':<8}")
print("-" * 50)
for i, d_in in enumerate(in_dims):
    w = model.params[i * 2]  # W1, W2, W3
    actual_std = w.std()
    he_std = np.sqrt(2.0 / d_in)
    print(f"Linear{i+1}({d_in:>3}→)  {actual_std:.5f}      {he_std:.5f}      {actual_std/he_std:.3f}")

## 9. 검증

In [ ]:
N = 8
x_t = np.random.randn(N, 784).astype(np.float32)

# task별 forward shape
for task, exp_dim in [("multiclass", 10), ("binary", 1), ("regression", 1)]:
    m = MLP(task=task, seed=0)
    logits = m.forward(x_t)
    assert logits.shape == (N, exp_dim), f"{task}: {logits.shape}"
    assert logits.dtype == np.float32

# params/grads 구조
m = MLP(task="multiclass", seed=0)
assert len(m.params) == 6
assert len(m.grads) == 6
assert m.params[0].shape == (784, 256)
assert m.params[2].shape == (256, 128)
assert m.params[4].shape == (128, 10)

# backward 후 grads 비 0
x_t = np.random.randn(N, 784).astype(np.float32)
t_t = np.eye(10, dtype=np.float32)[np.arange(N) % 10]
logits = m.forward(x_t)
m.backward(cross_entropy_grad(logits, t_t))
assert any(np.any(g != 0) for g in m.grads), "grads must be non-zero after backward"

# seed 재현성
m1 = MLP(task="multiclass", seed=7)
m2 = MLP(task="multiclass", seed=7)
assert np.allclose(m1.params[0], m2.params[0])

print("all MLP validation passed")

## 요약

이 노트북에서 확인한 내용은 다음과 같다.

| 항목 | 내용 |
|---|---|
| 구조 | `Sequential(Linear, Sigmoid, Linear, Sigmoid, Linear)` |
| 총 파라미터 수 | 235,146 (multiclass 기준), float32 ≈ 0.9 MB |
| `params` | `[W1, b1, W2, b2, W3, b3]` 6개 배열 |
| `forward` 출력 | `(N, output_dim)` raw logit (numpy float32) |
| `backward` 입력 | `d(loss)/d(logits)` from `losses.*_grad` |
| seed 파생 | 최상위 seed 1개 → 레이어별 seed 3개 자동 파생 |

**핵심 설계 원칙**
- `forward`는 raw logit만 반환한다. activation과 gradient 계산은 `losses.py`가 담당한다.
- `grads[i]`는 `params[i]`와 배열 참조를 공유하므로, backward 후 grads를 읽으면 항상 최신 gradient를 반환한다.
- optimizer는 `params`와 `grads`를 순회하며 `p -= lr * g`로 in-place 업데이트한다.